In [1]:
# Import required libraries
import os
import pandas as pd

In [2]:
# Load the dataset
df = pd.read_csv('../data/train_transcriptomics_SDY2867.tsv', sep='\t')
df = df.drop(columns=['transcriptomics_id'])
df.head()

,participant_id,timepoint,ensembl_gene_id,raw_count,tpm_count,material
0,SDY2867.SUB389725,0,ENSG00000000003,2.0,0.038336,Whole blood
1,SDY2867.SUB389727,-14,ENSG00000000003,10.0,0.200864,Whole blood
2,SDY2867.SUB389692,1,ENSG00000000003,16.0,0.964728,Whole blood
3,SDY2867.SUB389670,28,ENSG00000000003,17.0,0.670581,Whole blood
4,SDY2867.SUB389681,0,ENSG00000000003,2.0,0.094223,Whole blood


In [3]:
# raw_count has actual values (no nulls); material is uniformly 'Whole blood'
# SDY2867 has 5 timepoints: [-14, 0, 1, 7, 28] vs 2 for 2020_UGA
# Drop material before pivoting; drop raw_count to keep tpm_count only, consistent with 2019/2020 pipeline
print('raw_count null fraction:', df['raw_count'].isna().mean())
print('material unique values:', df['material'].unique())
print('timepoints:', sorted(df['timepoint'].unique()))
print('participants:', df['participant_id'].nunique())

raw_count null fraction: 0.0
material unique values: <StringArray>
['Whole blood']
Length: 1, dtype: str
timepoints: [np.int64(-14), np.int64(0), np.int64(1), np.int64(7), np.int64(28)]
participants: 73


In [4]:
# Drop material (entirely 'Whole blood'); drop raw_count to match pipeline (tpm_count only)
df = df.drop(columns=['raw_count', 'material'])
df.head()

,participant_id,timepoint,ensembl_gene_id,tpm_count
0,SDY2867.SUB389725,0,ENSG00000000003,0.038336
1,SDY2867.SUB389727,-14,ENSG00000000003,0.200864
2,SDY2867.SUB389692,1,ENSG00000000003,0.964728
3,SDY2867.SUB389670,28,ENSG00000000003,0.670581
4,SDY2867.SUB389681,0,ENSG00000000003,0.094223


In [5]:
df['timepoint'].unique()

array([  0, -14,   1,  28,   7])

In [6]:
# Filter to only timepoints 0, 7, 28. Timepoints -14 and 1 not in the challenge set
df = df[df['timepoint'].isin([0, 7, 28])]

# Pivot: each row is a participant_id, each gene+timepoint becomes a column
df_pivot = df.pivot_table(
    index='participant_id',
    columns=['timepoint', 'ensembl_gene_id'],
    values='tpm_count'
)

# Flatten multi-level columns to "d{timepoint}_{gene}" format
df_pivot.columns = [f'd{int(tp)}_{gene}' for tp, gene in df_pivot.columns]
df_pivot = df_pivot.reset_index()

print(df_pivot.shape)

(73, 164707)


In [7]:
df_pivot.head(1)

,participant_id,d0_ENSG00000000003,d0_ENSG00000000005,d0_ENSG00000000419,d0_ENSG00000000457,d0_ENSG00000000460,d0_ENSG00000000938,d0_ENSG00000000971,d0_ENSG00000001036,d0_ENSG00000001084,...,d28_ENSG00000310527,d28_ENSG00000310529,d28_ENSG00000310530,d28_ENSG00000310532,d28_ENSG00000310533,d28_ENSG00000310534,d28_ENSG00000310535,d28_ENSG00000310536,d28_ENSG00000310537,d28_ENSG00000310539
0,SDY2867.SUB389659,0.37521,0.0,37.867705,8.410112,1.897157,452.649272,1.293022,12.468978,16.280764,...,40.093229,0.0,0.0,0.0,4.03519,0.857613,7.787015,0.277979,0.379442,3.724106


In [8]:
# Save the cleaned dataset to the cleaned_data folder
os.makedirs('../cleaned_data', exist_ok=True)
df_pivot.to_csv('../cleaned_data/transcriptomics_SDY2867_cleaned.csv', index=False)